# Study 867 — Currency Crash Risk 💥

**Do high-carry currencies go up by the stairs and down by the elevator?**

Brunnermeier, Nagel & Pedersen (2008) argue the carry trade's premium is compensation
for **crash risk**: high-interest (carry) currencies are **negatively skewed** — gentle
appreciation while you earn the rate differential, punctuated by violent unwinds. The
higher the carry, the deeper the skew, and a long-high / short-low carry basket inherits
that crash tail. We test both halves on a weekly 8-currency tape vs USD
(2003-12-12 → 2026-06-26, 8 currencies incl. the notorious high-carry MXN).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: fixed current membership (no de-pegged legs) —
magnitudes are an upper bound.*


## 1. The idea in one picture

You earn the interest differential by holding a high-yield currency (MXN, NZD, AUD) funded in a low-yield one (JPY, CHF). Most weeks it drifts gently up — *the stairs*. Then a risk-off shock hits, everyone unwinds at once, and it gaps down — *the elevator*. That asymmetry is **negative skew**, and BNP say it deepens with the carry. So we (a) check whether higher-carry currencies are more negatively skewed and (b) measure the crash tail of the carry basket.

In [1]:
import numpy as np, pandas as pd
R = dict(basket_skew=-1.39, worst_week=-13.3, max_dd=-36.0, calm_ann=12.6, off_ann=-173.6, premium_ann=3.29, spearman=-0.833)
print('carry basket realized skew: %+.2f  (deeply negative = the crash tail)' % R['basket_skew'])
print('  worst single week %+.1f%%   max drawdown %+.1f%%' % (R['worst_week'], R['max_dd']))
print('  calm weeks %+.1f%%/yr  vs  worst-5%% weeks %+.1f%%/yr (annualised)' % (R['calm_ann'], R['off_ann']))
print('  premium you are paid for it: %+.2f%%/yr' % R['premium_ann'])
print('  higher carry -> more negative skew: Spearman rank corr %+.2f' % R['spearman'])

carry basket realized skew: -1.39  (deeply negative = the crash tail)
  worst single week -13.3%   max drawdown -36.0%
  calm weeks +12.6%/yr  vs  worst-5% weeks -173.6%/yr (annualised)
  premium you are paid for it: +3.29%/yr
  higher carry -> more negative skew: Spearman rank corr -0.83


## 2. Is the skew tied to the carry ordering? A live synthetic control

We plant the crash in a seeded toy world (`edge>0`: a fat negative factor tail whose loading rises with carry) and check the detector recovers a negative skew-carry slope — and stays *silent* on the null (`edge=0`, symmetric factor, no crash asymmetry). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from fx_crash import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=867, n_weeks=1000))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.02, seed=867, n_weeks=1000))
print('null world   : skew-carry slope %+.3f  basket skew %+.2f  (should be ~0)' % (null['slope'], null['basket_skew']))
print('planted world: skew-carry slope %+.3f  basket skew %+.2f  (should light up)' % (planted['slope'], planted['basket_skew']))

null world   : skew-carry slope +0.001  basket skew -0.05  (should be ~0)
planted world: skew-carry slope -0.277  basket skew -1.33  (should light up)


## 3. The honest verdict — real crash, but a weak/borderline signal and no paycheck

On the real tape the carry basket is deeply negatively skewed (**-1.39**, worst week -13.3%, max DD -36.0%) and higher carry clearly predicts more negative skew (Spearman **-0.83**, permutation *p* = 0.008) — the Brunnermeier-et-al signature is genuinely there and correctly signed. **But** the strict significance bar is not cleared: a Newey-West *t* on the basket's own skewness is only **-1.51** (the skew *t* is structurally low-powered against rare crashes). And the premium you are paid for the tail is weak (**+3.29%/yr**, *t* = +1.73), collapsing to **+0.71%/yr** after a modest borrow — pennies in front of a steamroller. **Signal: Weak. Tradability: Mirage.**